> **How to use this notebook.** Step 2 of 5. Needs: an internet connection; section 0 copies the Nurnet files from `data/nurnet/` and downloads every other raw input from its official source (`fetch_data.py`). To place the files by hand instead, follow [HOW_TO_USE.md, 'Linking the data yourself'](HOW_TO_USE.md#linking-the-data-yourself). Derived inputs: a prebuilt pack (`DERIVED_URL`, when public) or a rebuild (`REBUILD_DERIVED = True`, 15-45 min).
>
> **Nurnet files (included).** The site list (`nurnet_nuraghi.shp`, 7,167 nuraghi with tower counts) and the per-site building stone file are Nurnet's data, included in `data/nurnet/`; 02 copies them to the names the configuration expects (`DATA_DIR/sites/`, `DATA_DIR/stone/`) and checks their SHA-256. For the latest version and source information contact Nurnet at nurnet.presidente@gmail.com. Details: [HOW_TO_USE.md](HOW_TO_USE.md#nurnet-files).
>
> Order: 01_setup, 02_inputs, 03_scenarios, 04_analysis_report, 05_audit (Jupyter or Google Colab). Full guide: [notebooks/HOW_TO_USE.md](HOW_TO_USE.md).

# 02 Inputs: where the files are, how to load them, preflight

**What you need.** The raw inputs are NOT in the repository. Section 0 downloads every PUBLIC one from its official
source and rebuilds the files the model reads (`notebooks/fetch_data.py`); the Nurnet files are included in
`data/nurnet/` and copied (below). To place the files yourself, follow
[HOW_TO_USE.md, 'Linking the data yourself'](HOW_TO_USE.md#linking-the-data-yourself). Layout under `DATA_DIR`
(default `data/`), the layout the configs expect under `data_dir`:

```
data/
  sites/nurnet_nuraghi.{shp,shx,dbf,prj,cpg}                    nuraghe sites (Nurnet's data, copied from data/nurnet/)
  costsurface/sardinia_drag_cost_8dir_mu03_25832_int.tif        sledge-drag cost, 8 directions, mu 0.3 (derived from the DEM)
  costsurface/sardinia_slope_deg_25832.tif                      slope (sea-landing rule; derived from the DEM)
  geology/sardinia_lithology_25832.{shp,...}                    ARPAS 2019 lithology with STONE classes
  dem/sardinia_dem_GLO30_25832.tif                              Copernicus DEM GLO-30 (rebuilds + map outline)
  water/sardinia_rivers_25832.{shp,...}                         RAS rivers (wider-literature inputs only)
  soils/soils250k.geojson                                       soil map 1:250,000 (food-capacity inputs only)
  stone/nid_materiale_prod.csv                                  building stone per site (Nurnet's data, copied from data/nurnet/)
data/derived/cache/                                             derived inputs: haul tables, k-best quarry routes,
                                                                sea routes, ... (data/derived/README.md)
```

**Nurnet files.** The site list (nurnet_nuraghi.shp: 7,167 nuraghi with tower counts) and the per-site building
stone file are Nurnet's data; the files are included in `data/nurnet/` (`nurnet_nuraghi.{shp,shx,dbf,prj,cpg}`,
`nurnet_site_building_stone.csv`). Section 0 copies them to `DATA_DIR/sites/nurnet_nuraghi.*` and
`DATA_DIR/stone/nid_materiale_prod.csv` and checks their SHA-256 against `notebooks/inputs_manifest.csv`. For the
latest version and source information contact Nurnet at nurnet.presidente@gmail.com.

**Two ways to get the derived inputs** (section 0b): download the **prebuilt** derived-input pack of the full
audited run (`DERIVED_URL`; placeholder until the results package is public) into `data/derived/cache/`, or
**rebuild** all of them from the raw inputs (`REBUILD_DERIVED = True`; about 30-45 min with the hybrid stone rule,
~2.5 GB peak; `data/derived/README.md`). Section 6 can also rebuild only the base haul table.

Runtime (estimates): section 0 downloads about 400 MB (a few minutes) and builds the DEM, cost and slope rasters
(a few minutes, ~1.5 GB RAM); preflight with SHA-256 about a minute; the optional rebuild in section 6 a few
minutes and ~2.5 GB RAM.

## Config (the only cell you edit)

All paths are set at the top of this cell. With `"auto"` the notebook uses the repository layout: `code/`,
`config/`, `data/` (raw inputs, see `data/README.md`), `data/derived/cache/` (derived inputs, see
`data/derived/README.md`) and `outputs/` (everything the notebooks write). Environment variables `NB_REPO_DIR`,
`NB_DATA_DIR`, `NB_DERIVED_DIR`, `NB_CONFIG`, `NB_OUTPUTS_DIR`, `NB_WORK_DIR`, `NB_RESULTS_DIR`, `NB_LOCK` override
the defaults (for headless runs with `jupyter nbconvert --execute`).

**Google Colab:** the repository is cloned to `/content/Repast_Nuraghe` by the setup cell. Each notebook runs in its own Colab runtime, so put the data and outputs on Google Drive: `MOUNT_DRIVE = True`, `DATA_DIR = "/content/drive/MyDrive/Repast_Nuraghe/data"`, `DERIVED_DIR = "/content/drive/MyDrive/Repast_Nuraghe/data/derived/cache"`, `OUTPUTS_DIR = "/content/drive/MyDrive/Repast_Nuraghe/outputs"` (the same values in every notebook).

`STONE_RULE` (only for rebuilds, sections 0b and 6): `"hybrid"` (default, the rule of the full audited run; uses the
per-site building-stone file copied from `data/nurnet/` to `DATA_DIR/stone/`); `"stones7"` (nearest outcrop of the 7
permitted classes for every site; results differ); `"auto"` = hybrid if that file is present, else stones7. `PREFLIGHT` picks which file set is checked:
`"run"` (raw inputs + derived cache for model runs), `"rebuild"` (raw inputs for section 6).

`FETCH` (section 0): download the public raw inputs from their official sources and rebuild the model files
(`fetch_data.py`; files already present and correct are kept). `FETCH_RASTERS`: also build the DEM mosaic and the
cost and slope rasters. `DERIVED_URL`: URL of the prebuilt derived-input pack (placeholder: `None` until the
results package is public). `REBUILD_DERIVED`: rebuild every derived input instead (long).

In [ ]:
import os

# Repository layout: the only lines you may need to change. "auto" = the default folder of the repository.
REPO_DIR = os.environ.get("NB_REPO_DIR", "auto")         # the cloned repository ("auto": the parent of notebooks/)
REPO_URL = "https://github.com/maltaweel/Repast_Nuraghe"  # Colab: cloned to /content/Repast_Nuraghe if not found
DATA_DIR = os.environ.get("NB_DATA_DIR", "auto")         # raw inputs = data_dir ("auto": REPO_DIR/data; see data/README.md)
DERIVED_DIR = os.environ.get("NB_DERIVED_DIR", "auto")   # cached derived inputs ("auto": REPO_DIR/data/derived/cache)
CONFIG = os.environ.get("NB_CONFIG", "auto")             # base config ("auto": REPO_DIR/config/config_full_audited.yaml)
OUTPUTS_DIR = os.environ.get("NB_OUTPUTS_DIR", "auto")   # everything written here ("auto": REPO_DIR/outputs)
WORK_DIR = os.environ.get("NB_WORK_DIR", "auto")         # fast local disk for code + cache copies ("auto": OUTPUTS_DIR/work)
RESULTS_DIR = os.environ.get("NB_RESULTS_DIR", "auto")   # where runs are kept ("auto": OUTPUTS_DIR/notebook_runs)
MOUNT_DRIVE = False                                      # Colab only: mount Google Drive at /content/drive
INSTALL = True                                           # install OpenMPI + pinned packages if this runtime lacks them
LOCK = os.environ.get("NB_LOCK") or None                 # optional flock file (shared servers only)

STONE_RULE = os.environ.get("NB_STONE_RULE", "hybrid")  # "hybrid" (default, the full audited rule) | "stones7" | "auto"
PREFLIGHT = os.environ.get("NB_PREFLIGHT", "rebuild")    # "run" (derived cache present) | "rebuild" (raw inputs)
REBUILD = os.environ.get("NB_REBUILD", "") == "1"       # section 6: rebuild a haul table from the raw inputs

FETCH = os.environ.get("NB_FETCH", "1") == "1"          # section 0: download the public raw inputs (fetch_data.py)
FETCH_RASTERS = os.environ.get("NB_FETCH_RASTERS", "1") == "1"   # section 0: also build DEM, cost and slope rasters
DERIVED_URL = os.environ.get("NB_DERIVED_URL") or None  # section 0b: prebuilt derived-input pack (placeholder)
REBUILD_DERIVED = os.environ.get("NB_REBUILD_DERIVED", "") == "1"  # section 0b: rebuild every derived input

## 1. Setup
Same cell in every notebook: repository, folders, install check.

In [ ]:
# Setup (no edits needed): find the repository, set the folders, install if needed, import the helpers in repro.py.
import os, sys, glob
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and MOUNT_DRIVE and not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")

def _is_repo(d):
    return os.path.isfile(os.path.join(d, "notebooks", "repro.py")) and os.path.isdir(os.path.join(d, "code"))

def _find_repo(p):
    cands = [p] if p != "auto" else [os.path.dirname(os.getcwd()), os.getcwd(), "/content/Repast_Nuraghe"]
    for c in cands:
        if _is_repo(c):
            return os.path.abspath(c)
    if IN_COLAB and p == "auto":                     # Colab opens only the notebook: clone the repository once
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "/content/Repast_Nuraghe"], check=True)
        return "/content/Repast_Nuraghe"
    raise FileNotFoundError("Repository not found (tried " + ", ".join(cands) + "). Set REPO_DIR in the config cell.")

REPO_DIR = _find_repo(REPO_DIR)
PACKAGE_DIR = REPO_DIR                                     # name used by the helper calls below
sys.path.insert(0, os.path.join(REPO_DIR, "notebooks"))
import repro
if OUTPUTS_DIR == "auto":
    OUTPUTS_DIR = os.path.join(REPO_DIR, "outputs")
OUTPUTS_DIR = os.path.abspath(OUTPUTS_DIR)
L = repro.set_layout(repo=REPO_DIR, data=DATA_DIR, cache=DERIVED_DIR, config=CONFIG,
                     reference=os.path.join(OUTPUTS_DIR, "reference"))
DATA_DIR, DERIVED_DIR, CONFIG = L["data"], L["cache"], L["config"]
WORK_DIR = os.path.abspath(os.path.join(OUTPUTS_DIR, "work") if WORK_DIR == "auto" else WORK_DIR)
RESULTS_DIR = os.path.abspath(os.path.join(OUTPUTS_DIR, "notebook_runs") if RESULTS_DIR == "auto" else RESULTS_DIR)
for k in ("REPO_DIR", "DATA_DIR", "DERIVED_DIR", "CONFIG", "WORK_DIR", "RESULTS_DIR"):
    print(f"{k:12s}", globals()[k])
status = repro.ensure_env(install=INSTALL)
print("MPI:", repro.mpi_report(), "| Python", sys.version.split()[0], "| CPUs", os.cpu_count())
assert status == "ok", "Restart the kernel now (see message above), then run this notebook again."

## 0. Getting the data
Every input of the model, where it comes from, and how to get it. `fetch_data.py` (the next cell, `FETCH = True`)
does all of it automatically; the manual alternative gives the same files (step by step, with the folder tree and the
verification commands: [HOW_TO_USE.md, 'Linking the data yourself'](HOW_TO_USE.md#linking-the-data-yourself)).
`DATA_DIR` is the folder set in the config cell (default `data/`); commands run from the repository folder.

| input (file under `DATA_DIR`) | where it comes from | automatic (`fetch_data.py`) | manual alternative | SHA-256 = the run's copy? |
|---|---|---|---|---|
| nuraghe sites `sites/nurnet_nuraghi.{shp,shx,dbf,prj,cpg}` | Nurnet's data, **included** in this repository: `data/nurnet/nurnet_nuraghi.*` | copied | `mkdir -p DATA_DIR/sites && cp data/nurnet/nurnet_nuraghi.* DATA_DIR/sites/` | **yes** |
| building stone per site `stone/nid_materiale_prod.csv` | Nurnet's data, **included**: `data/nurnet/nurnet_site_building_stone.csv` | copied (renamed) | `mkdir -p DATA_DIR/stone && cp data/nurnet/nurnet_site_building_stone.csv DATA_DIR/stone/nid_materiale_prod.csv` | **yes** |
| lithology `geology/sardinia_lithology_25832.*` | ARPAS 2019, Carta Litologica 1:25,000 (Sardegna Geoportale): https://webgis.regione.sardegna.it/scaricocartografiaETL/arpas/carta_litologica_2019.zip | downloaded (243 MB, SHA-256 checked) + `tools_extra/build_lithology.py` | download the zip, then `python tools_extra/build_lithology.py carta_litologica_2019.zip DATA_DIR/geology/sardinia_lithology_25832.shp` | `.prj`, `.cpg` yes; `.shp/.shx/.dbf` no (same 47,780 polygons, IDs, units and vertices; 5 mm datum offset and row order of the run's copy) |
| DEM `dem/sardinia_dem_GLO30_25832.tif` | Copernicus DEM GLO-30, 8 tiles `Copernicus_DSM_COG_10_N{38..41}_00_E{008,009}_00_DEM.tif` from the open AWS bucket https://copernicus-dem-30m.s3.amazonaws.com/ (no login) + land mask: RAS regional boundary `limiteAmministrReg_CTR.zip` | downloaded (136 MB, SHA-256 per tile) + `tools_extra/build_dem.py` (`FETCH_RASTERS`) | download the tiles and the mask, then `python tools_extra/build_dem.py TILES_DIR DATA_DIR/dem/sardinia_dem_GLO30_25832.tif --mask region limiteAmministrRegionale_CTR.shp` | no (same recipe and grid; the run's mask layer, the regional catchments as held in a project database, is not public) |
| drag cost `costsurface/sardinia_drag_cost_8dir_mu03_25832_int.tif` | computed from the DEM | `code/tools/build_cost_mu.py` (`FETCH_RASTERS`) | `cd code && python tools/build_cost_mu.py CFG 0.3 DATA_DIR/costsurface/sardinia_drag_cost_8dir_mu03_25832_int.tif` (`CFG`: a config whose `data_dir` is `DATA_DIR`) | only if the DEM is identical |
| slope `costsurface/sardinia_slope_deg_25832.tif` | computed from the DEM | `tools_extra/build_slope.py` (`FETCH_RASTERS`) | `python tools_extra/build_slope.py DATA_DIR/dem/sardinia_dem_GLO30_25832.tif DATA_DIR/costsurface/sardinia_slope_deg_25832.tif` | only if the DEM is identical |
| soils `soils/soils250k.geojson` | Aru, Baldaccini & Vacca 1991 (RAS/AGRIS), WFS layer `Agris:carta_dei_suoli_sardegna_scala_250000`: https://geoserver.sardegnaportalesuolo.it/geoserver/wfs | downloaded (24 MB) | `curl -o DATA_DIR/soils/soils250k.geojson "<WFS GetFeature URL in HOW_TO_USE.md>"` | **yes** after writing back the GeoServer `timeStamp` of the run's copy (done automatically; by hand the file differs only in that field) |
| rivers `water/sardinia_rivers_25832.*` | RAS, PPR 2006 'Fiumi torrenti e altri corsi d'acqua' (Sardegna Geoportale): https://webgis.regione.sardegna.it/scaricocartografiaETL/ppr/fiumiTorrenti_ARC.zip | downloaded (10 MB, SHA-256 checked) + `tools_extra/build_rivers.py` | download the zip, then `python tools_extra/build_rivers.py fiumiTorrenti_ARC.zip DATA_DIR/water/sardinia_rivers_25832.shp` | `.prj`, `.cpg` yes; `.shp/.shx/.dbf` no (same 41,877 segments, IDs and vertices; 5 mm offset, row order) |

**Nurnet files.** The site list (nurnet_nuraghi.shp: 7,167 nuraghi with tower counts) and the per-site building stone
file are Nurnet's data; the files are included in `data/nurnet/`. For the latest version and source information
contact Nurnet at nurnet.presidente@gmail.com.

**Derived inputs** (haul tables, k-best quarry routes, sea tables, food catchments, cost rasters per friction; list:
`data/derived/README.md`): section 0b downloads the prebuilt pack of the full audited run (`DERIVED_URL`; placeholder
until the results package is public) or rebuilds them all from the raw inputs (`REBUILD_DERIVED = True`, i.e.
`python notebooks/fetch_data.py --rebuild-derived`; about 30-45 min with the hybrid stone rule, ~2.5 GB peak).

Files rebuilt from an official source are recorded in `DATA_DIR/fetch_provenance.json`; the preflight (section 2)
reports them as *rebuilt (not byte-comparable)* instead of *wrong*. Verify any file by hand with `sha256sum FILE` and
the value in `notebooks/inputs_manifest.csv`.

**Expected result of the next cell:** a table with the six Nurnet files, `soils` and the `.prj/.cpg` files
*identical*; rivers, lithology, DEM, cost and slope *rebuilt (not byte-comparable)*; `0 failed/differing`.

In [ ]:
import fetch_data
if FETCH:
    FETCH_ROWS = fetch_data.fetch(DATA_DIR, rasters=FETCH_RASTERS)
else:
    FETCH_ROWS = fetch_data.fetch(DATA_DIR, check_only=True)
bad_nurnet = [r["path"] for r in FETCH_ROWS if r["path"].startswith(("sites/", "stone/")) and r["status"] != "identical"]
assert not bad_nurnet, f"Nurnet files not identical to the run's copies: {bad_nurnet}. " + fetch_data.NURNET_NOTE

## 2. Preflight: presence + SHA-256 of every needed input
Reads `notebooks/inputs_manifest.csv` (path, size, SHA-256, what it is needed for) and checks each file. Raw inputs
are checked by size and SHA-256 (the values of the files used for the 31 Oct 2026 package); derived inputs and
the per-site stone file are checked for presence only. A missing or changed file stops here with a list.

In [ ]:
rows = repro.preflight(PACKAGE_DIR, need=PREFLIGHT, check_hash=True)
import pandas as pd
man = pd.DataFrame(repro.read_manifest(PACKAGE_DIR))
print(man.groupby("group").agg(files=("path", "size"), MB=("bytes", lambda b: round(pd.to_numeric(b, errors="coerce").sum() / 1e6, 1))))

## 3. What the inputs contain
The sites file and the cost raster, read the same way as the model does.

In [ ]:
import pyogrio, rasterio
s = pyogrio.read_dataframe(os.path.join(DATA_DIR, "sites", "nurnet_nuraghi.shp"), read_geometry=False)
print(len(s), "sites; fields:", ", ".join(s.columns[:20]))
print(s.TWR_N.value_counts().sort_index().rename("sites by tower count").to_string())
p = os.path.join(DATA_DIR, "costsurface", "sardinia_drag_cost_8dir_mu03_25832_int.tif")
if os.path.exists(p):
    with rasterio.open(p) as r:
        print(f"cost raster {r.width} x {r.height} cells, {r.count} bands (N, NE, E, SE, S, SW, W, NW), {r.res[0]:.0f} m, {r.crs}")

## 4. Stone rule: hybrid (the rule of the full audited run)
**Default: `hybrid`** (`stone_rule: hybrid` in `config/config_full_audited.yaml`): each nuraghe takes the **recorded
building stone** where it is known, otherwise the nearest outcrop (least drag cost) of the 7 permitted classes
(`allowed_stones`). The per-site file is **Nurnet's data (site building stone)**, included in this repository as
`data/nurnet/nurnet_site_building_stone.csv` and copied by section 0 to `DATA_DIR/stone/nid_materiale_prod.csv`
(UTF-8 CSV; the code reads the columns **`codice`** = Nurnet UID, e.g. `NID0052900`, and **`sa_materiale`** = the
recorded stone, e.g. `Basalto`, `Granito`, `Trachite e Calcare`; token map `materiale.map` in the config). For the
latest version and source information contact Nurnet at nurnet.presidente@gmail.com.

* `STONE_RULE = "stones7"`: every site uses the nearest outcrop of the 7 classes; results then differ from the full
  audited run.
* **Only rebuilding the haul tables reads that file** (`preprocess.py`). Model runs never do.

**Expected result:** `STONE RULE = hybrid (per-site building-stone file found: ..., columns OK)`.

In [ ]:
RULE = repro.materiale_status(PACKAGE_DIR, want=STONE_RULE)
print("rule used for any rebuild in this session:", RULE)

## 0b. Derived inputs: prebuilt pack or rebuild
The model runs read cached derived inputs (`DERIVED_DIR`; list: `data/derived/README.md`). Either download the
prebuilt pack of the full audited run (`DERIVED_URL`, placeholder until the results package is public) or rebuild
everything from the raw inputs (`REBUILD_DERIVED = True`). Rebuild time from the project's logs: about 30-45 min
with the hybrid stone rule (15-20 min with stones7), peak ~2.5 GB (one process at a time). The full audited run
reused derived inputs built by earlier runs of the project (data/derived/README.md, provenance note), so a
rebuild is not guaranteed byte-identical to the pack; 05_audit's rerun check is defined on the pack.

**Expected result:** `derived inputs in ...: 68 of 68 present`.

In [ ]:
if DERIVED_URL:
    fetch_data.fetch_derived(DERIVED_DIR, DERIVED_URL)
elif REBUILD_DERIVED:
    fetch_data.rebuild_derived(DATA_DIR, DERIVED_DIR, stone_rule=RULE)
else:
    print("DERIVED_URL not set and REBUILD_DERIVED False: using what is in DERIVED_DIR.")
    print(fetch_data.REBUILD_NOTES)
DERIVED_MISSING = fetch_data.derived_status(DERIVED_DIR)

## 5. Load the prebuilt cache (default)
Copies `code/` (verified) and the derived-input cache `DERIVED_DIR` (without the optional `cost_mu/`, which is
linked) to `WORK_DIR`, and writes `WORK_DIR/config.yaml` = `CONFIG` (default `config/config_full_audited.yaml`)
with only `data_dir`, `cache_dir`, `output_dir` and `stone_rule` set. No model parameter changes.

In [ ]:
CFG = repro.make_workdir(PACKAGE_DIR, WORK_DIR, stone_rule=RULE, cache="prebuilt")
import pandas as pd
h = pd.read_csv(os.path.join(WORK_DIR, "cache", "sites_haul.csv"))
print(len(h), "sites in the haul table; quarry source:", h.Q_SRC.value_counts().to_dict())
print("haul (m): median", h.haul_m.median().round(0), "mean", h.haul_m.mean().round(0))

## 6. Optional: rebuild a haul table from the raw inputs
`REBUILD = True` (or `NB_REBUILD=1`) runs the multi-source directional Dijkstra over the whole island
(`preprocess.py`), in a separate cache folder so the prebuilt one is not touched, and compares `sites_haul.csv`
and `site_stone_sets.csv` **byte for byte** with the prebuilt cache when that cache was built with the same rule
(a different rule differs by design: reported, not an error).

The complete rebuild of every cache product (cost rasters per friction, literature haul tables, k-best routes,
sea tables, food catchments, ...) is listed step by step in `data/derived/README.md`. It is not run here.

In [ ]:
if REBUILD:
    import filecmp, shutil
    RB = os.path.join(WORK_DIR, "rebuild")
    os.makedirs(os.path.join(RB, "cache"), exist_ok=True)
    cfg_rb = os.path.join(RB, "config.yaml")
    txt = open(CFG).read().replace(f"cache_dir: {os.path.join(WORK_DIR, 'cache')}", f"cache_dir: {os.path.join(RB, 'cache')}")
    txt = txt.replace(f"output_dir: {os.path.join(WORK_DIR, 'output')}", f"output_dir: {os.path.join(RB, 'output')}")
    open(cfg_rb, "w").write(txt)
    repro.sh([sys.executable, "preprocess.py", cfg_rb, "--rule", RULE], cwd=os.path.join(WORK_DIR, "code"), lock=LOCK)
    for rel in ("sites_haul.csv", "site_stone_sets.csv"):
        pre = os.path.join(repro.LAYOUT["cache"], rel)
        if not os.path.isfile(pre):
            print(f"{rel}: no prebuilt copy in {repro.LAYOUT['cache']} to compare with"); continue
        same = filecmp.cmp(os.path.join(RB, "cache", rel), pre, shallow=False)
        print(f"{rel} (rebuilt with {RULE}) == prebuilt copy (bytes): {same}"
              + ("" if same else "  <- expected only if the prebuilt cache used another stone rule"))
else:
    print("REBUILD is False: using the prebuilt cache (section 5).")

Inputs ready. Next: `03_scenarios.ipynb` (then 04, 05_audit).